In [ ]:
import tensorflow as tf
import os

if 'COLAB_GPU' in os.environ:
    MAIN_DIR_PATH = os.path.join("/", "content")
    LABELS_PATH = os.path.join(MAIN_DIR_PATH, "drive","MyDrive","datasets", "turker_scores_full_interview.csv")
    DRIVE_PATH = os.path.join(MAIN_DIR_PATH, 'drive' ,'MyDrive')
    SAVE_CHECKPOINT_PATH = os.path.join(DRIVE_PATH, 'model_checkpoints', 'ckpt_epoch_{epoch:02d}.keras')
    CHECKPOINTS_DIR = os.path.join(DRIVE_PATH, 'model_checkpoints')
    VIDEOS_FRAMES_PATH = os.path.join(MAIN_DIR_PATH,'dataset', "videos_frames")

    # Enable GPU memory growth to avoid OOM errors
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
    # Enable XLA (Accelerated Linear Algebra) for potential speedups
    tf.config.optimizer.set_jit(True)

    from google.colab import drive; drive.mount('/content/drive')
    !unzip -q /content/drive/MyDrive/datasets/videos_frames.zip -d /content/dataset/

elif os.path.exists('/kaggle'):
    MAIN_DIR_PATH = os.path.join("/", "kaggle", "working")
    LABELS_PATH= os.path.join(MAIN_DIR_PATH, 'turker_scores_full_interview.csv')
    SAVE_CHECKPOINT_PATH = os.path.join(MAIN_DIR_PATH, 'ckpt_epoch_{epoch:02d}.keras')
    CHECKPOINTS_DIR = os.path.join(
        "/", "kaggle", "input", "checkpoint", "tensorflow2", "default", "1")
    VIDEOS_FRAMES_PATH = os.path.join(MAIN_DIR_PATH,'dataset', "videos_frames")

    # Enable memory growth
    physical_devices = tf.config.list_physical_devices('GPU')
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Enable XLA (Accelerated Linear Algebra)
    tf.config.optimizer.set_jit(True)
    # Set graph optimizations
    tf.config.optimizer.set_experimental_options({
        "layout_optimizer": True,
        "constant_folding": True,
        "shape_optimization": True,
        "remapping": True,
        "arithmetic_optimization": True,
        "dependency_optimization": True,
        "loop_optimization": True,
        "function_optimization": True,
        "debug_stripper": True
    })

    !gdown 1C_YeHVXKrkKOPysJxG6sVMdESKSjaCv_
    !gdown 1e9km9KLY3qkVxlCutDBZnC49263dsd5F
    !unzip -q /kaggle/working/videos_frames.zip -d /kaggle/working/dataset
else:
    from hireverse.utils.utils import BASE_DIR
    MAIN_DIR_PATH =BASE_DIR
    LABELS_PATH= os.path.join(
        BASE_DIR, "data", "external", "turker_scores_full_interview.csv"
    )
    SAVE_CHECKPOINT_PATH = os.path.join(MAIN_DIR_PATH, "model_artifacts", "ckpt_epoch_{epoch:02d}.keras")
    CHECKPOINTS_DIR = os.path.join(
        MAIN_DIR_PATH,"model_artifacts")
    VIDEOS_FRAMES_PATH = os.path.join(MAIN_DIR_PATH,'data', 'processed','videos_frames')



In [15]:
import os
# MY_Chosen_LABELS = ['Colleague', 'Engaged', 'Excited', 'EyeContact', 'Smiled', 'Calm']
LABELS = ['Engaged', 'Calm', 'Friendly', 'Excited', 'Colleague', 'RecommendHiring']
participant_ids =['P1', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P20', 'P21', 'P22', 'P24', 'P25', 'P27', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P37', 'P42', 'P43', 'P44', 'P45', 'P47', 'P48', 'P49', 'P50', 'P52', 'P53', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62', 'P63', 'P64', 'P65', 'P66', 'P67', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P83', 'P84', 'P85', 'P86', 'P89', 'PP1', 'PP3', 'PP4', 'PP5', 'PP6', 'PP7', 'PP8', 'PP10', 'PP11', 'PP12', 'PP13', 'PP14', 'PP15', 'PP16', 'PP17', 'PP20', 'PP21', 'PP22', 'PP24', 'PP25', 'PP27', 'PP29', 'PP30', 'PP31', 'PP32', 'PP33', 'PP34', 'PP35', 'PP37', 'PP42', 'PP43', 'PP44', 'PP45', 'PP47', 'PP48', 'PP49', 'PP50', 'PP52', 'PP53', 'PP55', 'PP56', 'PP57', 'PP58', 'PP59', 'PP60', 'PP61', 'PP62', 'PP63', 'PP64', 'PP65', 'PP66', 'PP67', 'PP69', 'PP70', 'PP71', 'PP72', 'PP73', 'PP74', 'PP76', 'PP77', 'PP78', 'PP79', 'PP80', 'PP81', 'PP83', 'PP84', 'PP85', 'PP86', 'PP89']
INPUT_SIZE = (224, 224)
INFERENCE = True
LOAD_CHECKPOINT_PATH = os.path.join(CHECKPOINTS_DIR, "ckpt_epoch_08.keras")
checkpoint_path_exists = os.path.exists(LOAD_CHECKPOINT_PATH)

FRAMES_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 8
TEST_BATCH_SIZE = TRAIN_BATCH_SIZE*FRAMES_BATCH_SIZE
EPOCHS = 60  # 12 continuous hours of T4, 30 hours per week total

In [16]:
import os
import re
from typing import List, Tuple
import pandas as pd
import cv2
import numpy as np
from natsort import natsorted

class DatasetHandler:
    @staticmethod
    def get_labels_dict(participant_id: str):
        df = pd.read_csv(
            LABELS_PATH
        )
        df = df.loc[
            (df["Participant"] == participant_id.lower()) & (df["Worker"] == "AGGR")
        ]
        return df.iloc[0].to_dict()

    @staticmethod
    def get_participant_ids():
        p_participant_numbers, pp_participant_numbers = DatasetHandler._get_p_and_pp_participant_number()
        participant_ids = []
        for prefix, participant_numbers in [
            ("P", p_participant_numbers),
            ("PP", pp_participant_numbers),
        ]:
            for participant_number in participant_numbers:
                participant_id = f"{prefix}{participant_number}"
                participant_ids.append(participant_id)
        return participant_ids

    def get_participant_dir(participant_id):
        return os.path.join(VIDEOS_FRAMES_PATH, participant_id)

    @staticmethod
    def get_sorted_participant_frames_images(participant_id, is_image_greyscale=False) -> List[np.ndarray]:
        participant_dir = DatasetHandler.get_participant_dir(participant_id)
        frames = []
        for filename in natsorted(os.listdir(participant_dir)):
            if any(filename.endswith(ext) for ext in [".jpg", ".jpeg", ".png"]):
                image_path = os.path.join(participant_dir, filename)
                image = cv2.imread(image_path)
                if is_image_greyscale:
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                if image is not None:
                    frames.append(image)
        return frames

    @staticmethod
    def yield_sorted_participant_frames_images(participant_id, convert_to_greyscale=False):
        participant_dir = DatasetHandler.get_participant_dir(participant_id)
        for filename in natsorted(os.listdir(participant_dir)):
            if any(filename.endswith(ext) for ext in [".jpg", ".jpeg", ".png"]):
                image_path = os.path.join(participant_dir, filename)
                image = cv2.imread(image_path)
                if convert_to_greyscale:
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                if image is not None:
                    yield image

    @staticmethod
    def get_number_of_frames(participant_id):
        participant_dir = DatasetHandler.get_participant_dir(participant_id)
        return len(
            [
                f
                for f in os.listdir(participant_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
        )

In [17]:
# import os
# from PIL import Image
# import numpy as np
# from multiprocessing import Pool, cpu_count

# def preprocess_images_for_mobilenetv2(frames_path, target_size):
#     for filename in os.listdir(frames_path):
#         if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
#             filepath = os.path.join(frames_path, filename)
#             try:
#                 with Image.open(filepath) as img:
#                     resized_img = img.resize(target_size, Image.Resampling.LANCZOS)

#                     img_array = np.array(resized_img)
#                     if img_array.ndim == 2:
#                         img_array = np.expand_dims(img_array, axis=-1)
#                     elif img_array.ndim == 3 and img_array.shape[-1] != 1:
#                         raise ValueError(f"Image {filename} is not 1-channel grayscale")

#                     rgb_array = np.repeat(img_array, 3, axis=-1)
#                     final_img = Image.fromarray(rgb_array, 'RGB')
#                     final_img.save(filepath)

#             except Exception as e:
#                 print(f"Error processing {filepath}: {e}")

# def process_subdir(args):
#     subdir, target_size = args
#     preprocess_images_for_mobilenetv2(subdir, target_size)
#     print(f"Finished {subdir[-4:]}")

# if __name__ == "__main__":
#     all_subdirs = []
#     for root, dirs, _ in os.walk(VIDEOS_FRAMES_PATH):
#         all_subdirs.append(root)

#     args_list = [(subdir, INPUT_SIZE) for subdir in all_subdirs]

#     workers = min(cpu_count(), len(all_subdirs))

#     with Pool(workers) as pool:
#         pool.map(process_subdir, args_list)

In [18]:
# import os
# import random

# for participant_id in participant_ids:
#     participant_dir = DatasetHandler.get_participant_dir(participant_id)
#     c = 0
#     num_frames = random.randint(30, 60)

#     for f in os.listdir(participant_dir):
#         c += 1
#         if c > num_frames:
#             file_path = os.path.join(participant_dir, f)
#             os.remove(file_path)

In [ ]:
import gc
import cv2
import numpy as np
import time
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def participant_frames_generator(participant_id, number_of_frames_in_batch=FRAMES_BATCH_SIZE):
    total_frames = DatasetHandler.get_number_of_frames(participant_id)
    # Truncate to discard partial batches
    total_framesـafter_truncation = (total_frames // number_of_frames_in_batch) * number_of_frames_in_batch

    frame_generator = DatasetHandler.yield_sorted_participant_frames_images(
        participant_id, convert_to_greyscale=False
    )
    label_dict = DatasetHandler.get_labels_dict(participant_id)

    for _ in range(0, total_framesـafter_truncation, number_of_frames_in_batch):
        frames_batch = []
        for __ in range(number_of_frames_in_batch):
            frame = next(frame_generator)
            frame = tf.keras.applications.mobilenet_v2.preprocess_input(frame)   # [-1,1]
            frames_batch.append(frame)
        yield frames_batch, {comp: label_dict[comp] for comp in LABELS}

def participant_frames_generator_few_batches(participant_id, batch_size=TEST_BATCH_SIZE):
    total_frames = DatasetHandler.get_number_of_frames(participant_id)

    frame_generator = DatasetHandler.yield_sorted_participant_frames_images(
        participant_id, convert_to_greyscale=False
    )
    label_dict = DatasetHandler.get_labels_dict(participant_id)

    frames_batch = []
    for i in range(total_frames):
        frame = next(frame_generator)
        frame = tf.keras.applications.mobilenet_v2.preprocess_input(frame)
        frames_batch.append(frame)

        # When batch is full, yield it
        if len(frames_batch) == batch_size:
            yield frames_batch, {comp: label_dict[comp] for comp in LABELS}
            frames_batch = []

    # Yield any leftover frames (non-truncated)
    if frames_batch:
        yield frames_batch, {comp: label_dict[comp] for comp in LABELS}

def train_generator(train_ids, participants_per_batch=TRAIN_BATCH_SIZE):
    while True:
        shuffled_ids = np.random.permutation(train_ids)
        # Initialize generators and track remaining batches per participant
        participant_gens = {}
        batches_left = {}
        for pid in shuffled_ids:
            total_frames = DatasetHandler.get_number_of_frames(pid)
            num_batches = total_frames // FRAMES_BATCH_SIZE
            if num_batches > 0:
                participant_gens[pid] = participant_frames_generator(pid, FRAMES_BATCH_SIZE)
                batches_left[pid] = num_batches
        active_participants = list(batches_left.keys())

        # Generate batches until no participants can form a full batch
        while len(active_participants) > 0:
            selected_pids = np.random.choice(
                active_participants,
                size=participants_per_batch,
                replace=len(active_participants) < participants_per_batch
            )

            X_batch = []
            y_batch = {comp: [] for comp in LABELS}
            exhausted_pids = set()

            for pid in selected_pids:
                if batches_left[pid] <= 0:
                    continue  # Skip if no batches left (due to replacement)
                try:
                    frames, labels = next(participant_gens[pid])
                    X_batch.extend(frames)
                    for comp in LABELS:
                        y_batch[comp].extend([labels[comp]] * len(frames))
                    batches_left[pid] -= 1
                    if batches_left[pid] == 0:
                        exhausted_pids.add(pid)
                except StopIteration:
                    exhausted_pids.add(pid)

            for pid in exhausted_pids:
                if pid in active_participants:
                    active_participants.remove(pid)

            # expected_frames = participants_per_batch * FRAMES_BATCH_SIZE
            # if len(X_batch) == expected_frames:
            X_batch = np.array(X_batch)
            indices_list = np.random.permutation(len(X_batch))
            # print(f"\nyield batch having numbe_r of frames: {len(X_batch)}")
            yield X_batch[indices_list], {comp: np.array(y_batch[comp])[indices_list] for comp in LABELS} # returns a shuffled version of the entire X_batch, not just a single item.
            # elif not active_participants:  # ✅ Only break if no participants left
            #     print(f"\n skipped batch having number of frames: {len(X_batch)}")
            #     break

In [20]:
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(participant_ids[:int(len(participant_ids)/2)], test_size=0.2, random_state=42)

train_ids = natsorted(train_ids + [f"P{pid}" for pid in train_ids])
test_ids = natsorted(test_ids + [f"P{pid}" for pid in test_ids])
# train_gen = train_generator(train_ids)


In [21]:
import tensorflow as tf

policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

In [22]:
from math import ceil

total_number_of_train_frames = sum(DatasetHandler.get_number_of_frames(pid) for pid in train_ids)
number_of_frame_batches = sum(
    DatasetHandler.get_number_of_frames(pid) // FRAMES_BATCH_SIZE
    for pid in train_ids
)
steps_per_epoch = ceil(number_of_frame_batches / TRAIN_BATCH_SIZE )
discarded_frames = total_number_of_train_frames - steps_per_epoch * FRAMES_BATCH_SIZE* TRAIN_BATCH_SIZE
print(discarded_frames)
print(f"{(100 * discarded_frames / total_number_of_train_frames):.1f}%")

770
0.1%


In [23]:
from tensorflow.keras import layers, models

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

def create_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)

    # Base model
    base_model = MobileNetV2(
        input_tensor=inputs,
        weights='imagenet',
        include_top=False,
        pooling='avg'  # Better than GlobalAveragePooling
    )

    # Strategic unfreezing (top 20 layers trainable)
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False


    x = base_model.output
    x = layers.Dense(256)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.3)(x)
    outputs = [layers.Dense(1, activation='linear', name=comp)(x) for comp in LABELS]

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Learning rate schedule with warmup
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-5,
        decay_steps=steps_per_epoch*EPOCHS,
        warmup_target=1e-3,
        warmup_steps=steps_per_epoch*3
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss={comp: tf.keras.losses.MeanSquaredError() for comp in LABELS},

    )
    return model


height, width = INPUT_SIZE
input_shape = (height, width, 3)
model = create_model(input_shape)

/var/folders/d8/gg3vwxbd3k54_36zkft8f2_c0000gn/T/ipykernel_84886/3219179269.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


In [24]:
loaded_model = tf.keras.models.load_model(LOAD_CHECKPOINT_PATH)
print("Loaded model summary:")
loaded_model.summary()

fresh_model = create_model(input_shape)
print("\nFresh model summary:")
fresh_model.summary()

/Users/bassel27/personal_projects/hireverse/venv/lib/python3.9/site-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(


Loaded model summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_5 (Cast)       │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ cast_5[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis

 Total params: 5,351,383 (20.41 MB)

...

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,587,462 (9.87 MB)

 Trainable params: 1,381,958 (5.27 MB)

 Non-trainable params: 1,205,504 (4.60 MB)

In [25]:
#  Total params: 5,351,383 (20.41 MB)
#  Trainable params: 1,381,958 (5.27 MB)
#  Non-trainable params: 1,205,504 (4.60 MB)
#  Optimizer params: 2,763,921 (10.54 MB)

import tensorflow as tf
import re

callbacks = [
      tf.keras.callbacks.EarlyStopping(
          monitor='loss',
          patience=5,
          min_delta=0.001
      ),
      tf.keras.callbacks.ModelCheckpoint(
          filepath=SAVE_CHECKPOINT_PATH,
          verbose = 1,
          save_weights_only=False,
          save_freq='epoch',
          monitor='loss',
          save_best_only=True
      )
  ]

initial_epoch = 0
if checkpoint_path_exists:
    match = re.search(r'epoch_(\d+)', LOAD_CHECKPOINT_PATH)
    if match:
        initial_epoch = int(match.group(1))
    model = tf.keras.models.load_model(LOAD_CHECKPOINT_PATH)
if not INFERENCE:
  history = model.fit(
      train_generator(train_ids),
      steps_per_epoch=steps_per_epoch,
      epochs=EPOCHS,
      callbacks = callbacks,
      initial_epoch=initial_epoch,
  )
  model.save('final_model.keras')

# Evaluation

In [33]:
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np
from tqdm import tqdm
from collections import defaultdict

video_data = defaultdict(lambda: {'frame_preds': [], 'video_true': None})
c= True
for pid in tqdm(test_ids[:], desc="Predicting videos"):
    predictions_per_competency = defaultdict(list)

    for frames_batch, labels in participant_frames_generator_few_batches(pid):
        frames_batch_np = np.array(frames_batch)  # Shape: (batch_size, height, width, 3)
        batch_preds = model.predict(frames_batch_np, verbose=0)  # List of 6 arrays, each (batch_size, 1)

        for i, comp in enumerate(LABELS):
            predictions_per_competency[comp].extend(batch_preds[i].flatten())   # each comp inside predictions_per_comp is of shape (batch_size)
        video_data[pid]['video_true'] = np.array([labels[comp] for comp in LABELS])

    video_data[pid]['frame_preds'] = np.array([np.mean(predictions_per_competency[comp]) for comp in LABELS])   # shape: (6,)

video_true = []
video_pred = []

for pid in video_data:
    video_true.append(video_data[pid]['video_true'])
    video_pred.append(video_data[pid]['frame_preds'])

video_true = np.array(video_true)
video_pred = np.array(video_pred)

Predicting videos:  86%|████████▌ | 24/28 [45:58<07:39, 114.92s/it]


KeyboardInterrupt: 

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

print(video_pred)
print(video_true)
for i, comp in enumerate(LABELS):
    true_vals = video_true[:, i]    # all true values for the current label
    pred_vals = video_pred[:, i]

    # Pearson r
    r, _ = pearsonr(true_vals, pred_vals)

    # R^2 score
    r2 = r2_score(true_vals, pred_vals)

    print(f"{comp}: Pearson r = {r:.4f}, R² = {r2:.4f}")

[[6.926 5.496 6.8   6.965 4.805 4.977]]
[[5.54137978 5.35107543 5.25478356 5.04389015 5.33300425 5.10622414]]


ValueError: x and y must have length at least 2.

# Test on Random Data

In [ ]:
import numpy as np
import tensorflow as tf

# Assuming INPUT_SIZE and LABELS are defined

# Create 3 random images with shape (height, width, 3)
random_images = np.random.randint(0, 256, size=(3, *INPUT_SIZE, 3), dtype=np.uint8)

# Preprocess all images (convert to float32 and apply MobileNetV2 preprocessing)
random_images = tf.keras.applications.mobilenet_v2.preprocess_input(random_images.astype(np.float32))

# random_images shape is now (3, height, width, 3) — ready as a batch

# Predict for the batch
predictions = model.predict(random_images)

# predictions shape depends on your model's output, assuming it's a list of arrays per label
# If your output is a list of arrays (one per label), each with shape (batch_size, 1),
# you can iterate like this:

for label, preds_per_label in zip(LABELS, predictions):
    print(f"Predictions for {label}:")
    for i, pred in enumerate(preds_per_label):
        print(f"  Image {i+1}: {pred[0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Predictions for Engaged:
  Image 1: 6.390625
  Image 2: 6.22265625
  Image 3: 6.3359375
Predictions for Calm:
  Image 1: 5.0
  Image 2: 5.0234375
  Image 3: 4.78125
Predictions for Friendly:
  Image 1: 6.3203125
  Image 2: 6.33203125
  Image 3: 6.27734375
Predictions for Excited:
  Image 1: 5.9375
  Image 2: 5.8046875
  Image 3: 5.8828125
Predictions for Colleague:
  Image 1: 5.234375
  Image 2: 5.15625
  Image 3: 5.1953125
Predictions for RecommendHiring:
  Image 1: 4.94921875
  Image 2: 4.9140625
  Image 3: 4.87890625


In [ ]:
# Run this to see actual values flowing through your pipeline
sample_pid = 'P1'
labels = DatasetHandler.get_labels_dict(sample_pid)
print("Labels dict:", labels)
print("Specific label values:")
for comp in LABELS:
    print(f"{comp}: {labels[comp]} (type: {type(labels[comp])})")

Labels dict: {'Participant': 'p1', 'Worker': 'AGGR', 'Overall': 5.29731562336, 'RecommendHiring': 5.10622414048, 'Colleague': 5.33300424956, 'Engaged': 5.54137978101, 'Excited': 5.04389015411, 'EyeContact': 5.86611859187, 'Smiled': 3.57615974017, 'SpeakingRate': 4.86558979541, 'NoFillers': 3.77166473066, 'Friendly': 5.25478355985, 'Paused': 5.80046790524, 'EngagingTone': 5.14790937729, 'StructuredAnswers': 4.89158004615, 'Calm': 5.35107543307, 'NotStressed': 5.35075959402, 'Focused': 5.84522637357, 'Authentic': 5.61051274415, 'NotAwkward': 5.47753366771, 'Total': 93.1311955077}
Specific label values:
Engaged: 5.54137978101 (type: <class 'float'>)
Calm: 5.35107543307 (type: <class 'float'>)
Friendly: 5.25478355985 (type: <class 'float'>)
Excited: 5.04389015411 (type: <class 'float'>)
Colleague: 5.33300424956 (type: <class 'float'>)
RecommendHiring: 5.10622414048 (type: <class 'float'>)
